# Dynamic Model Router — Quickstart

Routes each LLM call to the cheapest model that can handle it — before any API call is made.

**3-layer cascade:**
- **Layer 1** — keyword heuristics (< 1 ms)
- **Layer 3** — ML classifier (frozen MiniLM + MLP head, ~10 ms)
- **Layer 2** — Gemini Flash Lite fallback (only when confidence is low)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manthanvaghela/dynamic-model-router/blob/main/examples/quickstart.ipynb)

## 1. Install

In [ ]:
# Core package (no ML layer)
!pip install -q dynamic-model-router

# With ML Layer 3 (recommended — adds sentence-transformers + sklearn)
# !pip install -q 'dynamic-model-router[ml]'

## 2. Set your API key

In [ ]:
import os

# Set your Google API key (free tier at aistudio.google.com)
os.environ['GOOGLE_API_KEY'] = 'your-key-here'

# Or Anthropic:
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

# Or OpenAI:
# os.environ['OPENAI_API_KEY'] = 'sk-...' 

## 3. Zero-config classify

In [ ]:
from classifier import classify

tasks = [
    "What is 2 + 2?",
    "Write a Python function to merge two sorted lists.",
    "Design a distributed rate limiter for 100K req/s with Redis.",
]

for task in tasks:
    d = classify(task)
    print(f"[{d.tier.value.upper():6}] {d.model_name:30} | {task[:60]}")

## 4. Router class — per-instance config

In [ ]:
from classifier import Router

# Layer 3 (ML) disabled — faster, no sentence-transformers needed
router = Router(layer3_enabled=False)
d = router.classify("Summarise this earnings call transcript")
print(d.tier.value, d.model_name, d.layer_used)

## 5. Estimate cost before calling the LLM

In [ ]:
router = Router()

tasks = [
    "Hello, how are you?",
    "Write a REST API in FastAPI with JWT auth.",
    "Design a CQRS architecture for a healthcare record system handling 10M patients.",
]

print(f"{'Task':55} {'Tier':8} {'Model':30} {'Est USD/call'}")
print("-" * 110)
for task in tasks:
    info = router.estimate_cost(task)
    print(f"{task[:55]:55} {info['tier']:8} {info['model']:30} ${info['est_usd_per_call']:.7f}")

## 6. Custom keywords — domain vocabulary injection

In [ ]:
from classifier import Router, KeywordPack
from classifier.core.types import TaskType

# Build a legal keyword pack
legal_pack = (
    KeywordPack.builder("legal")
    .add(TaskType.REASONING, ["indemnification", "precedent", "tortfeasor", "arbitration"])
    .add(TaskType.DOC_CREATION, ["clause", "agreement", "non-compete", "NDA"])
    .build()
)

router = Router(extra_keyword_packs=[legal_pack])
d = router.classify("Draft an indemnification clause for a vendor agreement")
print(d.tier.value, "|", d.task_type.value, "|", d.model_name)

## 7. Domain presets

In [ ]:
# Healthcare preset — HIPAA PII patterns + clinical vocabulary
router = Router.from_preset("healthcare")

clinical_tasks = [
    "What are the contraindications for metformin?",
    "65-year-old with chest pain and elevated troponin — differential diagnosis?",
    "Design a comprehensive CV risk reduction strategy for metabolic syndrome patient.",
]

for task in clinical_tasks:
    d = router.classify(task)
    flag = " [PII]" if d.compliance_flag else ""
    print(f"[{d.tier.value.upper():6}] {d.model_name:30}{flag}")
    print(f"         {task[:80]}\n")

## 8. @route_model decorator — wrap existing functions

In [ ]:
from classifier import route_model

@route_model(provider="google")
def call_llm(task: str, model_name: str = "gemini-2.5-flash"):
    """model_name is injected by the router — no manual classify() needed."""
    print(f"Would call model: {model_name}")
    return model_name

# Low complexity → gemini-2.5-flash, High complexity → gemini-2.5-pro
call_llm("Hello")
call_llm("Design a distributed consensus algorithm for Byzantine fault tolerance")

## 9. Evaluate routing accuracy on a labeled dataset

In [ ]:
# Using the built-in legal sample dataset
import subprocess
result = subprocess.run(
    ["dmr", "eval", "--data", "classifier/data/legal_tasks.jsonl", "--show-errors"],
    capture_output=True, text=True
)
print(result.stdout)

## 10. LangChain integration

In [ ]:
# pip install langchain-google-genai
from classifier.integrations.langchain import get_chat_model, DynamicChatModel

# Pattern A — one-shot per task
llm = get_chat_model("Translate this medical note to Spanish", provider="google")
# response = llm.invoke("Translate: Patient presents with acute chest pain")

# Pattern B — one agent, all tiers
dynamic_llm = DynamicChatModel(provider="google")
# chain = dynamic_llm | StrOutputParser()
# chain.invoke("Translate: Patient presents with acute chest pain")

print("LangChain integration ready — uncomment to make live calls.")

## 11. CrewAI integration

In [ ]:
# pip install crewai
from classifier.integrations.crewai import pick_llm_for_task, DynamicLLM

# Pattern A
# llm = pick_llm_for_task("Analyse Q3 earnings report", provider="google")
# agent = Agent(role="Analyst", goal="...", llm=llm)

# Pattern B — single agent, dynamic tier per call
# agent = Agent(role="Analyst", goal="...", llm=DynamicLLM(provider="google"))

print("CrewAI integration ready — uncomment to make live calls.")

## 12. AutoGen integration

In [ ]:
# pip install pyautogen
from classifier.integrations.autogen import get_autogen_llm_config, DynamicModelRouter

# Pattern A
task = "Analyse quarterly revenue and flag anomalies"
config = get_autogen_llm_config(task, provider="openai")
print("AutoGen config:", config['config_list'][0]['model'])

# Pattern B — reusable router
dyn_router = DynamicModelRouter(provider="openai")
for t in ["Hello", "Write a compiler"]:
    cfg = dyn_router.llm_config(t)
    print(f"  {t[:30]:30} → {cfg['config_list'][0]['model']}")

## 13. Google ADK integration

In [ ]:
# pip install google-adk
# from classifier.integrations.adk import dynamic_model_selector
# from google.adk.agents import LlmAgent
#
# agent = LlmAgent(
#     name="clinical_qa",
#     model="gemini-2.5-flash",
#     before_model_callback=dynamic_model_selector,
#     instruction="You are a clinical knowledge assistant.",
# )

print("ADK integration ready — see examples/adk_healthcare/ for full agents.")

## 14. Train Layer 3 on your own data

In [ ]:
# Your JSONL needs: {"task": "...", "task_type": "reasoning", "complexity": "standard"}

router = Router()
# metadata = router.train(
#     data="classifier/data/legal_tasks.jsonl",
#     max_iter=600,
# )
# print(metadata)

# Or via CLI:
# !dmr train --data classifier/data/legal_tasks.jsonl

print("Train your domain classifier with router.train('my_data.jsonl')")

## 15. Load from YAML config

In [ ]:
# Generate a starter config:
# !dmr init
#
# Then load it:
# router = Router.from_yaml("dmr.yaml")

# Minimal inline YAML example
import yaml, tempfile, os
cfg = yaml.dump({
    "providers": ["google"],
    "layer1_enabled": True,
    "layer3_enabled": False,
})
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(cfg)
    tmp = f.name

router = Router.from_yaml(tmp)
d = router.classify("Draft an email to a customer")
print(d.tier.value, d.model_name, "layer:", d.layer_used)
os.unlink(tmp)